# Run PRCA Survey on Google Colab with Google Drive Auto-Backup
This notebook will mount your Google Drive, pull the latest code from GitHub, install Ollama, and run the survey. Results are continuously backed up to your Drive.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama pandas tqdm requests

In [ ]:
import os
# Remove existing folder if re-running cell
!rm -rf SurveyResponder
!git clone https://github.com/DhruvKithany/SurveyResponder.git
%cd SurveyResponder


In [ ]:
import subprocess
import time

# Start Ollama server in the background
print('Starting Ollama server...')
subprocess.Popen(['ollama', 'serve'])

print('Waiting for server to start...')
time.sleep(5)

# Pull the llama3.1 model
print('Pulling llama3.1 model (this may take a few minutes)...')
!ollama pull llama3.1:latest
print('Model pulled successfully!')

In [ ]:
from google.colab import drive
import os
import pandas as pd
from SurveyResponder import SurveyResponder

# 1. Mount Google Drive
print('Mounting Google Drive... (You will be prompted to authorize access)')
drive.mount('/content/drive')

# 2. Initialize the Responder
responder = SurveyResponder(
    questions_path="prca_questions.json",
    persona_path="persona.json",
    model_name="llama3.1:latest",
    num_responses=10,
    temperature=1.0
)

# 3. Create a unique folder in your Google Drive based on model and temperature
base_drive_dir = '/content/drive/MyDrive/SurveyResponses'
model_safe = responder.model_name.replace(':', '_')
run_folder_name = f"run_{model_safe}_temp_{responder.temperature}"
run_dir = os.path.join(base_drive_dir, run_folder_name)
os.makedirs(run_dir, exist_ok=True)

output_file = os.path.join(run_dir, "prca_results.csv")
print(f"\nGenerating responses and saving continuously to:\n{run_dir}")

# 4. Run the survey! 
# SurveyResponder natively writes the `_response_log.csv` file after EVERY SINGLE QUESTION.
# Because `output_file` points directly into your mounted Google Drive, it will automatically 
# flush each question response straight into the cloud as it's generated, fully protecting 
# you from instance failures!
responder.run_write(output_file)
print("Done!")

# Load the final compiled responses
df = pd.read_csv(output_file)
display(df.head())